In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path.cwd().parent / "data" / "spy_eod_202303.txt")
df.columns = [k.strip(' ') for k in df.columns]
df

: 

In [ ]:
selected = df[df["[STRIKE]"] == 130.0]
print(selected)

In [19]:
quotes = pd.to_datetime(df["[QUOTE_READTIME]"].unique())
print(quotes.min(), quotes.max())
strikes = df["[STRIKE]"].unique()
expiries = df["[EXPIRE_DATE]"].unique()

print('num of quotes:', len(quotes))
print('num strikes:', len(strikes))
print('num expiries:', len(expiries))

groups = df.groupby(["[QUOTE_READTIME]"])

for quote_time, group in groups:
    print(len(group["[STRIKE]"].unique()))
    print(len(group["[EXPIRE_DATE]"].unique()))
    print()

2023-03-01 16:00:00 2023-03-31 16:00:00
num of quotes: 23
num strikes: 289
num expiries: 53
289
32

289
34

289
33

289
33

289
33

289
33

289
34

289
33

289
33

289
33

289
33

289
33

289
32

289
32

289
32

289
32

289
32

289
31

288
31

288
31

288
31

288
32

288
31



In [4]:
df = pd.DataFrame(columns=strikes, index=expiries)
for expiry, row in df.iterrows():
    for strike in row.index:
        count = 0
        for quote_time, group in groups:
            if expiry in group["[EXPIRE_DATE]"].values and strike in group["[STRIKE]"].values:
                count += 1
        pct = count / len(groups)
        df.loc[expiry, strike] = pct

df

,320.0,330.0,336.0,337.0,338.0,339.0,340.0,341.0,342.0,343.0,...,501.0,502.0,503.0,155.0,165.0,175.0,715.0,120.0,130.0,140.0
2023-03-01,0.043478,0.043478,0.043478,0.043478,0.043478,0.043478,0.043478,0.043478,0.043478,0.043478,...,0.043478,0.043478,0.043478,0.043478,0.043478,0.043478,0.043478,0.043478,0.043478,0.043478
2023-03-02,0.086957,0.086957,0.086957,0.086957,0.086957,0.086957,0.086957,0.086957,0.086957,0.086957,...,0.086957,0.086957,0.086957,0.086957,0.086957,0.086957,0.086957,0.086957,0.086957,0.086957
2023-03-03,0.130435,0.130435,0.130435,0.130435,0.130435,0.130435,0.130435,0.130435,0.130435,0.130435,...,0.130435,0.130435,0.130435,0.130435,0.130435,0.130435,0.130435,0.130435,0.130435,0.130435
2023-03-06,0.173913,0.173913,0.173913,0.173913,0.173913,0.173913,0.173913,0.173913,0.173913,0.173913,...,0.173913,0.173913,0.173913,0.173913,0.173913,0.173913,0.173913,0.173913,0.173913,0.173913
2023-03-07,0.217391,0.217391,0.217391,0.217391,0.217391,0.217391,0.217391,0.217391,0.217391,0.217391,...,0.217391,0.217391,0.217391,0.217391,0.217391,0.217391,0.217391,0.217391,0.217391,0.217391
2023-03-08,0.26087,0.26087,0.26087,0.26087,0.26087,0.26087,0.26087,0.26087,0.26087,0.26087,...,0.26087,0.26087,0.26087,0.26087,0.26087,0.26087,0.26087,0.26087,0.26087,0.26087
2023-03-09,0.304348,0.304348,0.304348,0.304348,0.304348,0.304348,0.304348,0.304348,0.304348,0.304348,...,0.304348,0.304348,0.304348,0.304348,0.304348,0.304348,0.304348,0.304348,0.304348,0.304348
2023-03-10,0.347826,0.347826,0.347826,0.347826,0.347826,0.347826,0.347826,0.347826,0.347826,0.347826,...,0.347826,0.347826,0.347826,0.347826,0.347826,0.347826,0.347826,0.347826,0.347826,0.347826
2023-03-13,0.391304,0.391304,0.391304,0.391304,0.391304,0.391304,0.391304,0.391304,0.391304,0.391304,...,0.391304,0.391304,0.391304,0.391304,0.391304,0.391304,0.391304,0.391304,0.391304,0.391304
2023-03-14,0.434783,0.434783,0.434783,0.434783,0.434783,0.434783,0.434783,0.434783,0.434783,0.434783,...,0.434783,0.434783,0.434783,0.434783,0.434783,0.434783,0.434783,0.434783,0.434783,0.434783


In [10]:
pairs = []
for expiry, row in df.iterrows():
    for strike in row.index:
        if df.loc[expiry, strike] == 1.0:
            pairs.append((expiry, strike))

expiries = np.array([x[0] for x in pairs])
strikes = np.array([x[1] for x in pairs])
np.save(Path.cwd().parent / "nn_learning" / "expiries_being_used.npy", expiries)
np.save(Path.cwd().parent / "nn_learning" / "strikes_being_used.npy", strikes)
print("expiries")
print(expiries)
print("strikes")
print(strikes)
print()
print('num of pairs that appear everywhere:', len(pairs))

print('num of pairs that dont appear everywhere:', len(df)*len(df.columns) - len(pairs))

expiries
[' 2023-03-31' ' 2023-03-31' ' 2023-03-31' ... ' 2025-12-19' ' 2025-12-19'
 ' 2025-12-19']
strikes
[320. 330. 336. ... 120. 130. 140.]

num of pairs that appear everywhere: 5472
num of pairs that dont appear everywhere: 9845


In [15]:
# get earliest quote
quotes_fixed = [quote[1:] for quote in quotes]
quotes_fixed = pd.to_datetime(quotes_fixed)
print(quotes_fixed.min())
print(quotes_fixed.max())

2023-03-01 16:00:00
2023-03-31 16:00:00
